# Model 10: Transfer Learning with ResNet18

## Architecture
- **Base**: ResNet18 pre-trained on ImageNet (14 million images)
- **Feature Extractor**: All layers frozen (no training)
- **Classifier**: Replaced with new fully connected layer for 10 classes
- **Fine-tuning**: After training the classifier, unfreeze all layers

In [ ]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import torchvision.models as models
import pickle
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from data_loader import AnimalsDataLoader
from train import ModelTrainer

print("✅ All imports successful!")

In [ ]:
# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model configuration
MODEL_NAME = 'model_10_transfer'
NUM_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 0.001
FINE_TUNE_LR = 0.0001

print(f"\nModel: {MODEL_NAME}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs (Stage 1): {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Fine-tune LR: {FINE_TUNE_LR}")
print(f"Base: ResNet18 pre-trained on ImageNet")

In [ ]:
# Load data
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

data_loader = AnimalsDataLoader(
    data_dir='../data/animals',
    batch_size=BATCH_SIZE,
    image_size=224,
    use_english_names=True
)

train_loader, val_loader, test_loader = data_loader.load_data(
    val_ratio=0.15,
    test_ratio=0.15
)

print("\n✓ Data loaded successfully!")
print(f"  Training: {len(train_loader.dataset)} images ({len(train_loader)} batches)")
print(f"  Validation: {len(val_loader.dataset)} images ({len(val_loader)} batches)")
print(f"  Test: {len(test_loader.dataset)} images ({len(test_loader)} batches)")

In [ ]:
# Create model (Feature Extractor Mode)
print("\n" + "="*60)
print("CREATING TRANSFER LEARNING MODEL")
print("="*60)

# Load pre-trained ResNet18
model = models.resnet18(pretrained=True)

#Freeze all layers (feature extractor mode)
for param in model.parameters():
    param.requires_grad = False

#Replace the classifier for 10 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, NUM_CLASSES)

model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nModel Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Frozen parameters: {frozen_params:,} (pre-trained features)")
print(f"  Trainable parameters: {trainable_params:,} (new classifier)")
print(f"  Training mode: Feature Extractor (only classifier trains)")

print(f"\n{model}")

In [ ]:
print("\n" + "="*60)
print("INITIALIZING TRAINER")
print("="*60)

trainer = ModelTrainer(
    model=model,
    device=device,
    model_name=MODEL_NAME,
    save_dir='../models_pickles'
)

print(f"✓ Trainer initialized")
print(f"  Save directory: {trainer.save_dir}")

In [ ]:
#Train Feature Extractor
print("\n" + "="*60)
print("STAGE 1: TRAINING CLASSIFIER")
print("="*60)
print("  - Feature extractor mode (all layers frozen)")
print("  - Only training the new classifier layer")
print("="*60)

history1 = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.0,
    patience=5
)

print(f"\n✅ Stage 1 Complete!")
print(f"  Best Validation Accuracy: {trainer.best_val_acc:.2f}%")
print(f"  Best Epoch: {trainer.best_epoch}")

In [ ]:
#Fine-tuning Entire Model
print("\n" + "="*60)
print("STAGE 2: FINE-TUNING ENTIRE MODEL")
print("="*60)
print("  - Unfreezing all layers")
print("  - Training entire model with lower learning rate")
print("="*60)

#Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# Update total trainable params
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters now: {trainable_params:,} (all layers)")

# Update trainer for fine-tuning with lower LR
trainer.model = model

history2 = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10,  # Fewer epochs for fine-tuning
    learning_rate=FINE_TUNE_LR,
    weight_decay=0.0001,  # L2 regularization
    patience=3
)

print(f"\n✅ Stage 2 Complete!")
print(f"  Best Validation Accuracy: {trainer.best_val_acc:.2f}%")
print(f"  Best Epoch: {trainer.best_epoch}")

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

test_results = trainer.evaluate(test_loader)

print(f"\nTest Results:")
print(f"  Accuracy: {test_results['accuracy']:.2f}%")
print(f"  Total samples tested: {len(test_results['true_labels'])}")

In [ ]:
# Plot training history
print("\n" + "="*60)
print("TRAINING VISUALIZATION")
print("="*60)

# Combine both stages
combined_history = {
    'train_loss': history1['train_loss'] + history2['train_loss'],
    'val_loss': history1['val_loss'] + history2['val_loss'],
    'train_acc': history1['train_acc'] + history2['train_acc'],
    'val_acc': history1['val_acc'] + history2['val_acc']
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(combined_history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(combined_history['val_loss'], label='Val Loss', linewidth=2)
axes[0].axvline(x=len(history1['train_loss'])-0.5, color='red', linestyle='--', alpha=0.5, label='Stage 1 → Stage 2')
axes[0].set_title(f'Loss During Training - {MODEL_NAME}', fontsize=14)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(combined_history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(combined_history['val_acc'], label='Val Acc', linewidth=2)
axes[1].axvline(x=len(history1['train_loss'])-0.5, color='red', linestyle='--', alpha=0.5, label='Stage 1 → Stage 2')
axes[1].set_title(f'Accuracy During Training - {MODEL_NAME}', fontsize=14)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
os.makedirs('../outputs', exist_ok=True)
plt.savefig(f'../outputs/{MODEL_NAME}_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training history saved to: outputs/{MODEL_NAME}_history.png")

In [ ]:
# Save results
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

results = {
    'model_name': MODEL_NAME,
    'model_class': 'ResNet18',
    'test_accuracy': test_results['accuracy'],
    'best_val_acc': trainer.best_val_acc,
    'best_epoch': trainer.best_epoch,
    'total_epochs': len(combined_history['train_loss']),
    'history': combined_history,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'fine_tune_lr': FINE_TUNE_LR,
        'epochs_stage1': EPOCHS,
        'epochs_stage2': 10,
        'num_classes': NUM_CLASSES,
        'total_params': total_params,
        'trainable_params': trainable_params,
        'frozen_params': frozen_params,
        'model_type': 'resnet18',
        'pretrained': 'ImageNet'
    },
    'device': str(device)
}

json_path = f'../models_pickles/{MODEL_NAME}_results.json'
os.makedirs('../models_pickles', exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to: {json_path}")

pickle_path = f'../models_pickles/{MODEL_NAME}.pkl'
if os.path.exists(pickle_path):
    size_mb = os.path.getsize(pickle_path) / (1024 * 1024)
    print(f"✓ Model pickle saved: {pickle_path} ({size_mb:.2f} MB)")
else:
    print("❌ Model pickle not found!")

In [ ]:
# Summary
print("\n" + "="*60)
print("SUMMARY - MODEL 10: TRANSFER LEARNING")
print("="*60)

print(f"\n📊 Performance:")
print(f"  Best Validation Accuracy: {trainer.best_val_acc:.2f}% (Epoch {trainer.best_epoch})")
print(f"  Test Accuracy: {test_results['accuracy']:.2f}%")

print(f"\n📈 Training Stats:")
print(f"  Total Epochs Trained: {len(combined_history['train_loss'])}")
print(f"  Stage 1 (Feature Extractor): {len(history1['train_loss'])} epochs")
print(f"  Stage 2 (Fine-tuning): {len(history2['train_loss'])} epochs")
print(f"  Final Train Accuracy: {combined_history['train_acc'][-1]:.2f}%")
print(f"  Final Val Accuracy: {combined_history['val_acc'][-1]:.2f}%")

print(f"\n🔍 Transfer Learning Analysis:")
print(f"  Total parameters: {total_params:,}")
print(f"  Frozen (pre-trained): {frozen_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  Pre-trained on: ImageNet (14M images)")

print(f"\n💾 Saved Files:")
print(f"  Model Pickle: models_pickles/{MODEL_NAME}.pkl")
print(f"  Results JSON: models_pickles/{MODEL_NAME}_results.json")
print(f"  Training Plot: outputs/{MODEL_NAME}_history.png")

print(f"\n🏆 Transfer Learning complete! This should be your best model!")